# LU-STW IETE Revision — Colab Runner (TITR-2026-0877)

Clones the repo, installs deps, then runs revision experiments once `R_s` / `R_t` are loaded.

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/Atulpatel123123123/LU-STW-IETE-revision.git"
DIR = "/content/LU-STW-IETE-revision"
if not os.path.isdir(DIR):
    subprocess.check_call(["git", "clone", REPO, DIR])
else:
    subprocess.check_call(["git", "-C", DIR, "pull"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", f"{DIR}/requirements.txt"])
sys.path.insert(0, DIR)
print("Ready:", DIR)

## Load overlapping-user matrices

Replace the stub below with your MovieLens / Yelp / Amazon loaders so that `R_s` and `R_t` are dense `numpy` arrays with the **same number of rows** (overlapping users).

In [ ]:
import numpy as np

# TODO: load real data
# from src.revision_experiments import build_movielens_disjoint
# R_s, R_t, meta = build_movielens_disjoint(ratings_df, movies_df)

R_s = R_t = None  # set these before running the next cell
assert R_s is not None and R_t is not None, "Load R_s and R_t first"
assert R_s.shape[0] == R_t.shape[0]
print(R_s.shape, R_t.shape)

In [ ]:
import json, os
from src.revision_experiments import (
    REVISION_VARIANTS, cv_report, confidence_tercile_gain,
    make_folds, run_variant, ranking_at_k
)

OUT = "/content/LU-STW-IETE-revision/results"
os.makedirs(OUT, exist_ok=True)

# Quick smoke: 1 seed, 50% density, core variants
smoke = {k: REVISION_VARIANTS[k] for k in ["LU-STW", "Procrustes", "User-only eta", "Target Only"]}
report = cv_report(R_s, R_t, smoke, densities=(0.5,), seeds=(0,), n_splits=5)
serial = {f"{a}|{b}": v for (a, b), v in report.items()}
with open(f"{OUT}/smoke_cv.json", "w") as f:
    json.dump(serial, f, indent=2)

terc = confidence_tercile_gain(R_s, R_t)
with open(f"{OUT}/confidence_terciles.json", "w") as f:
    json.dump(terc, f, indent=2)

R_tr, R_val = make_folds(R_t, 0.5, 5, 0)[0]
full = run_variant(R_s, R_tr, R_val)
rank = {}
for K in (5, 10):
    rank.update(ranking_at_k(full["R_hat"], R_tr, R_val, K=K))
with open(f"{OUT}/ranking_smoke.json", "w") as f:
    json.dump(rank, f, indent=2)

print("Wrote", OUT)
print(serial)
print("terciles", terc)
print("ranking", rank)